# Notebook 04 — Tables 2 & 3: Factor Spanning Tests and H–L Comparison

**Data needed:**
- `Results/cluster_month_panels_K_50/cluster_month_panel_K_50_lambda_{λ}.csv`
  (baseline + Vanilla proxy λ=0.001 + Inertia proxy λ=1000)
- `data/factor_returns.csv`

**Output:** `Table2_Alpha_Diagnostics.{csv,tex}`, `Table3_HL_Comparison.{csv,tex}`

In [1]:
import os, sys, warnings
import numpy as np
import pandas as pd
import statsmodels.api as sm
warnings.filterwarnings("ignore")

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, REPO_ROOT)

from utils.data_utils import (
    load_cluster_panel, load_cluster_ranking, pivot_and_rank,
    nw_tstat, stars, ann_sharpe, save_table, DATA_DIR,
)

NW_LAGS = 3
ranking = load_cluster_ranking()

# Factor model definitions — column names as they appear after loading
# (see Cell 2 for exact column name mapping)
FACTOR_MODELS = {
    "CAPM":  ["MKT"],
    "FF3":   ["MKT", "SMB", "HML"],
    "FFC":   ["MKT", "SMB", "HML", "Mom"],
    "FF5":   ["MKT", "SMB", "HML", "RMW", "CMA"],
    "FFPS":  ["MKT", "SMB", "HML", "PS_VWF"],
    "DHS":   ["MKT", "PEAD", "FIN"],
    "Q":     ["MKT", "R_ME", "R_IA", "R_ROE", "R_EG"],
    "SYM":   ["MKT", "SY_SMB", "MGMT", "PERF"],
    "PCA":   ["PCA_1", "PCA_2", "PCA_3", "PCA_4", "PCA_5"],
    "IPCA":  ["IPCA1", "IPCA2", "IPCA3", "IPCA4", "IPCA5"],
    "RPPCA": ["RPPCA_1", "RPPCA_2", "RPPCA_3", "RPPCA_4", "RPPCA_5"],
}

In [5]:
# ── Load and merge all factor data (matches reference notebook exactly) ────────
#
# All_factors_JFE.csv  : year, month columns; values in %; some cols to drop
# IPCA_factors.csv     : date col YYYYMM format; columns '1'-'5'
# RP-PCA Factors 2.csv : date col 'Mon-YY' format; values in %
#
# All three are merged on (year, month) and a datetime index is built.

DATA_PATH = os.path.join(REPO_ROOT, "data")   # replication_repo/data/
# If your Data folder is elsewhere, set this:
# DATA_PATH = "/ssd1/songjiangliu/shared/asset_clustering/Data"

# ── 1. JFE factors ────────────────────────────────────────────────
JFE = pd.read_csv(os.path.join(DATA_PATH, "factor_returns.csv"))

JFE["PS_VWF"] = JFE["PS_VWF"] * 100   # already scaled to pct in file; ×100 -> bps? 
                                        # reference does this, so we match it exactly

drop_cols = ["cap_risk", "INDPRO", "PCE",
             "z_longs", "z_shorts", "z_longs_diff", "z_shorts_diff", "ZLp", "ZSp"]
JFE.drop(columns=[c for c in drop_cols if c in JFE.columns], inplace=True)

# Divide all factor columns (everything except year, month, RF) by 100
factor_cols_jfe = [c for c in JFE.columns if c not in ("year", "month", "RF")]
JFE[factor_cols_jfe] = JFE[factor_cols_jfe] / 100

print(f"JFE factors: {JFE.shape}  cols: {list(JFE.columns)}")

# ── 2. IPCA factors ───────────────────────────────────────────────
IPCA = pd.read_csv(os.path.join(DATA_PATH, "IPCA_factors.csv"))
if "Unnamed: 0" in IPCA.columns:
    IPCA.drop(columns=["Unnamed: 0"], inplace=True)
IPCA.rename(columns={"1":"IPCA1","2":"IPCA2","3":"IPCA3","4":"IPCA4","5":"IPCA5"},
            inplace=True)
IPCA["year"]  = IPCA["date"].astype(str).str[:4].astype(int)
IPCA["month"] = IPCA["date"].astype(str).str[4:6].astype(int)
IPCA.drop(columns=["date"], inplace=True)
print(f"IPCA factors: {IPCA.shape}  cols: {list(IPCA.columns)}")

# ── 3. RP-PCA factors ─────────────────────────────────────────────
RP = pd.read_csv(os.path.join(DATA_PATH, "RP-PCA Factors.csv"))

def parse_rppca_date(date_str):
    month, year = date_str.split("-")
    year = int(year)
    year = 1900 + year if year >= 63 else 2000 + year
    return pd.to_datetime(f"{month} {year}", format="%b %Y")

RP["parsed_date"] = RP["date"].apply(parse_rppca_date)
RP["year"]        = RP["parsed_date"].dt.year
RP["month"]       = RP["parsed_date"].dt.month
RP.drop(columns=["date", "parsed_date"], inplace=True)
RP_numeric = [c for c in RP.columns if c not in ("year","month")]
RP[RP_numeric] = RP[RP_numeric] / 100
# Rename columns to RPPCA_1 … RPPCA_5
# PCA_1-PCA_5 keep their names; rename the 5 non-PCA columns to RPPCA_1-RPPCA_5
rp_other_cols = [c for c in RP.columns if c not in ("year","month") and not c.startswith("PCA_")]
rename_rp = {c: f"RPPCA_{i+1}" for i, c in enumerate(rp_other_cols)}
RP.rename(columns=rename_rp, inplace=True)
print(f"RP-PCA factors: {RP.shape}")
print(f"  PCA cols  : {[c for c in RP.columns if c.startswith('PCA_')]}")
print(f"  RPPCA cols: {[c for c in RP.columns if c.startswith('RPPCA_')]}")

# ── 4. Merge all three ────────────────────────────────────────────
PCA_all = IPCA.merge(RP, on=["year","month"])
factors_raw = JFE.merge(PCA_all, on=["year","month"])

# Build datetime index from year + month
factors_raw["date"] = pd.to_datetime(
    factors_raw["year"].astype(str) + "-" + factors_raw["month"].astype(str).str.zfill(2)
)
factors = factors_raw.set_index("date").sort_index()
factors.drop(columns=["year","month"], errors="ignore", inplace=True)

print(f"\nMerged factors: {factors.shape}")
print(f"Columns: {list(factors.columns)}")
print(f"Period : {factors.index[0].date()}  ->  {factors.index[-1].date()}")

JFE factors: (696, 22)  cols: ['MKT', 'SMB', 'HML', 'RMW', 'CMA', 'RF', 'year', 'month', 'Mom', 'PS_VWF', 'R_ME', 'R_IA', 'R_ROE', 'R_EG', 'PEAD', 'FIN', 'SY_SMB', 'MGMT', 'PERF', 'LN_F1', 'LN_F2', 'LN_F3']
IPCA factors: (527, 7)  cols: ['IPCA1', 'IPCA2', 'IPCA3', 'IPCA4', 'IPCA5', 'year', 'month']
RP-PCA factors: (650, 12)
  PCA cols  : ['PCA_1', 'PCA_2', 'PCA_3', 'PCA_4', 'PCA_5']
  RPPCA cols: ['RPPCA_1', 'RPPCA_2', 'RPPCA_3', 'RPPCA_4', 'RPPCA_5']

Merged factors: (492, 35)
Columns: ['MKT', 'SMB', 'HML', 'RMW', 'CMA', 'RF', 'Mom', 'PS_VWF', 'R_ME', 'R_IA', 'R_ROE', 'R_EG', 'PEAD', 'FIN', 'SY_SMB', 'MGMT', 'PERF', 'LN_F1', 'LN_F2', 'LN_F3', 'IPCA1', 'IPCA2', 'IPCA3', 'IPCA4', 'IPCA5', 'RPPCA_1', 'RPPCA_2', 'RPPCA_3', 'RPPCA_4', 'RPPCA_5', 'PCA_1', 'PCA_2', 'PCA_3', 'PCA_4', 'PCA_5']
Period : 1977-01-01  ->  2017-12-01


In [6]:
# ── Load cluster returns and align with factors ────────────────────
df  = load_cluster_panel(K=50, lam=1_000_000)
cr, _ = pivot_and_rank(df, lam=1_000_000, ranking_df=ranking)
print(f"Cluster returns: {cr.shape}")

common = cr.index.intersection(factors.index)
cr_c   = cr.loc[common]
fd_c   = factors.loc[common]
print(f"Common period  : {common[0].date()}  ->  {common[-1].date()}  ({len(common)} months)")

# Show which factor model columns are available
print()
for model, cols in FACTOR_MODELS.items():
    avail   = [c for c in cols if c in fd_c.columns]
    missing = [c for c in cols if c not in fd_c.columns]
    status  = "OK" if not missing else f"MISSING: {missing}"
    print(f"  {model:6s}  {status}")

Cluster returns: (511, 50)
Common period  : 1977-01-01  ->  2017-12-01  (486 months)

  CAPM    OK
  FF3     OK
  FFC     OK
  FF5     OK
  FFPS    OK
  DHS     OK
  Q       OK
  SYM     OK
  PCA     OK
  IPCA    OK
  RPPCA   OK


In [9]:
# ── Time-series regression with NW standard errors ────────────────
def ts_reg(y, X_df, lags=3):
    # Align on index explicitly, drop any rows where either y or X has NaN
    combined = pd.concat([y.rename("y"), X_df], axis=1, join="inner").dropna()
    if len(combined) < 24:
        return np.nan, np.nan, np.nan, np.nan
    y_ = combined["y"].values
    X_ = sm.add_constant(combined.drop(columns="y").values)
    res = sm.OLS(y_, X_).fit(cov_type="HAC", cov_kwds={"maxlags": lags})
    return res.params[0], res.bse[0], res.tvalues[0], res.rsquared

# ── Table 2: Alpha diagnostics across all factor models ────────────
table2_rows = {}

for model, cols in FACTOR_MODELS.items():
    avail = [c for c in cols if c in fd_c.columns]
    if not avail:
        print(f"  Skipping {model}: no columns available in factor data")
        continue
    if len(avail) < len(cols):
        print(f"  {model}: using {avail} (missing {set(cols)-set(avail)})")

    alphas, r2s, mean_rets = [], [], []
    for cluster in cr_c.columns:
        a, _, _, r2 = ts_reg(cr_c[cluster], fd_c[avail], lags=NW_LAGS)
        alphas.append(a)
        r2s.append(r2)
        mean_rets.append(cr_c[cluster].mean())

    alphas    = np.array(alphas)
    mean_rets = np.array(mean_rets)
    fin       = np.isfinite(alphas)

    sh2 = (np.mean(alphas[fin]) / np.std(alphas[fin])) ** 2           if np.std(alphas[fin]) > 0 else np.nan

    table2_rows[model] = {
        "A|alpha_i|":     round(np.mean(np.abs(alphas[fin])), 4),
        "A|alpha_i/r_i|": round(np.mean(np.abs(alphas[fin] / (mean_rets[fin] + 1e-10))), 4),
        "R2":              round(np.mean(np.array(r2s)[fin]), 4),
        "Sh2(alpha)":     round(sh2, 4),
    }

table2 = pd.DataFrame(table2_rows).T
table2.index.name = "Model"
print("\n=== Table 2: Alpha Diagnostics ===")
print(table2.to_string())
save_table(table2, "Table2_Alpha_Diagnostics")

InvalidIndexError: Reindexing only valid with uniquely valued Index objects

In [10]:
# ── Table 3: H-L portfolio comparison ─────────────────────────────
def hl_row(name, low_r, high_r, hl_r, lags=3):
    m_l,  _, t_l  = nw_tstat(low_r,  lags=lags)
    m_h,  _, t_h  = nw_tstat(high_r, lags=lags)
    m_hl, _, t_hl = nw_tstat(hl_r,   lags=lags)
    sr_hl = m_hl / hl_r.std() if hl_r.std() > 0 else np.nan
    return {
        "Model":    name,
        "Low":      f"{m_l:.4f}{stars(t_l)}",
        "t_Low":    f"({t_l:.2f})",
        "High":     f"{m_h:.4f}{stars(t_h)}",
        "t_High":   f"({t_h:.2f})",
        "H-L":      f"{m_hl:.4f}{stars(t_hl)}",
        "t_HL":     f"({t_hl:.2f})",
        "SR(H-L)":  f"{sr_hl:.4f}",
    }

rows3 = []

# Our model (K=50, lambda=1e6)
rows3.append(hl_row("Our Model",
                    cr.iloc[:, 0],           # L01
                    cr.iloc[:, -1],          # L50
                    cr.iloc[:, -1] - cr.iloc[:, 0]))

# Vanilla (lambda=0.001, near-zero regularisation)
# Inertia (lambda=1000, strong regularisation toward previous clusters)
for lam_v, lam_str_v, label_v in [
    (1e-3,   "lambda_0.001", "Vanilla"),
    (1000.0, "lambda_1000",  "Inertia"),
]:
    try:
        df_v    = load_cluster_panel(K=50, lam=lam_v)
        cr_v, _ = pivot_and_rank(df_v, lam=lam_v,
                                  lam_str=lam_str_v,
                                  ranking_df=ranking)
        idx = cr_v.index.intersection(cr.index)
        rows3.append(hl_row(label_v,
                            cr_v.loc[idx].iloc[:,  0],
                            cr_v.loc[idx].iloc[:, -1],
                            cr_v.loc[idx].iloc[:, -1] - cr_v.loc[idx].iloc[:, 0]))
        print(f"  {label_v}: loaded OK  ({len(idx)} months)")
    except Exception as e:
        print(f"  {label_v}: unavailable ({e})")

# Factor-based long-short benchmarks (from fd_c)
for fcol, fname in [
    ("MKT",    "Market"),
    ("SMB",    "Size"),
    ("HML",    "Value"),
    ("CMA",    "Investment"),
    ("Mom",    "Momentum"),
    ("PS_VWF", "Liquidity (PS)"),
    ("PEAD",   "PEAD"),
    ("FIN",    "Financing"),
]:
    if fcol not in fd_c.columns:
        continue
    f = fd_c[fcol].dropna()
    rows3.append(hl_row(fname,
                        f.clip(upper=0),
                        f.clip(lower=0),
                        f))

table3 = pd.DataFrame(rows3).set_index("Model")
print("\n=== Table 3: H-L Comparison ===")
print(table3.to_string())
save_table(table3, "Table3_HL_Comparison")
print("\nTables 2 and 3 saved.")

  Vanilla: loaded OK  (511 months)
  Inertia: loaded OK  (511 months)

=== Table 3: H-L Comparison ===
                       Low     t_Low       High   t_High        H-L    t_HL SR(H-L)
Model                                                                              
Our Model       -0.0192***   (-3.47)  0.0159***   (4.44)  0.0350***  (7.79)  0.4024
Vanilla         -0.0139***   (-2.92)  0.0111***   (3.49)  0.0250***  (6.32)  0.3197
Inertia          -0.0121**   (-2.57)  0.0176***   (5.40)  0.0296***  (8.19)  0.4608
Market          -0.0140***  (-10.10)  0.0203***  (17.71)  0.0063***  (3.06)  0.1435
Size            -0.0097***  (-12.17)  0.0119***  (12.80)    0.0022*  (1.68)  0.0770
Value           -0.0093***  (-10.95)  0.0119***  (10.77)    0.0026*  (1.72)  0.0897
Investment      -0.0061***  (-12.39)  0.0086***  (11.54)  0.0025***  (2.61)  0.1289
Momentum        -0.0119***   (-7.44)  0.0181***  (13.67)  0.0063***  (3.01)  0.1417
Liquidity (PS)  -0.0110***  (-11.43)  0.0153***  (14.81) 